# Course 04 lab — company, project, and feature requirements
Use the AI-2048 fixture to separate ownership from consumption, validate delegated specialization, protect authority boundaries, and trace policy change.

In [ ]:
import importlib.util, sys
from dataclasses import replace
from datetime import date
from pathlib import Path
spec = importlib.util.spec_from_file_location('course04_lab', Path('lab.py'))
lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
scenario = lab.load_scenario()
report = lab.resolve_project()
assert report.gate is lab.Gate.READY
print('gate:', report.gate.value)
print('applicable:', [item.id for item in report.applicable])
print(lab.evaluation_metrics(report, scenario['manifest'], scenario['graph']))

## 1 — Inspect decision rights
The meaning owner changes the obligation. Delegated owners choose named local controls. Enforcement owners operate the mechanism that observes or blocks behavior.

In [ ]:
for requirement in report.applicable:
    delegated = [(item.field, item.owner, item.agent_authority.value) for item in requirement.delegated_controls]
    enforcement = [(item.plane, item.owner) for item in requirement.enforcement]
    print(requirement.id, 'meaning_owner=', requirement.meaning_owner)
    print('  delegated=', delegated)
    print('  enforcement=', enforcement)

## 2 — Reject a weakening disguised as specialization
ARCH-031 selects a delegated review mechanism while preserving every fixed human-review control. ARCH-032 tries to make human review optional and must stop.

In [ ]:
requirements = {item.id: item for item in scenario['requirements']}
unsafe = lab.load_specialization(lab.SCENARIO_ROOT / 'project' / 'architecture' / 'ARCH-032-weakening.json')
weakening = lab.validate_specialization(unsafe, requirements[unsafe.parent_requirement_id])
codes = {item.code for item in weakening}
assert {'SPECIALIZATION_FIELD_NOT_DELEGATED', 'SPECIALIZATION_WEAKENS_PARENT'} <= codes
print(*weakening, sep='\n')

## 3 — Instructions cannot expand authority
A repository agent may read policy and propose project changes, but it cannot rewrite central policy, write an exception record, or turn a required control into an option.

In [ ]:
boundary_findings = lab.validate_write_paths(('catalog/enterprise/ai-governance/AI-030.json', 'exceptions/EXC-014.json'), scenario['boundary'])
instruction_findings = lab.validate_agent_instruction('AGENT-UNSAFE', {'human_review': 'optional'}, report.applicable)
assert all(item.severity is lab.Severity.ERROR for item in boundary_findings + instruction_findings)
print(*boundary_findings, *instruction_findings, sep='\n')

## 4 — Exceptions need independent, current authority
The selected exception is scoped and version-bound. Self-approval, the wrong authority, an old parent revision, or expiry turns it into a stop state.

In [ ]:
exception = scenario['exceptions'][0]
invalid = replace(exception, approver=exception.requester, expires_on=date(2026, 1, 1))
exception_findings = lab.validate_exception(invalid, requirements, scenario['manifest'].project, date(2026, 9, 20))
assert {'EXCEPTION_SELF_APPROVED', 'EXCEPTION_EXPIRED'} <= {item.code for item in exception_findings}
print(*exception_findings, sep='\n')

## 5 — Propagate an upstream change
AI-030 v4 adds a new fixed obligation. Traverse the relationship graph so direct and transitive consumers re-evaluate their claims rather than silently inheriting stale assurance.

In [ ]:
updated = lab.load_requirement(lab.SCENARIO_ROOT / 'updates' / 'AI-030-v4.json')
types = {'ARCH-031': 'project', 'REQ-REN-004': 'feature', 'ReviewServiceAdapter': 'implementation', 'TEST-REVIEW-004': 'evidence'}
impact = lab.analyze_impact(requirements['AI-030'], updated, scenario['graph'], types, scenario['specializations'][0])
predicted = [item.artifact_id for item in impact.impacted]
gold = ['ARCH-031', 'REQ-REN-004', 'ReviewServiceAdapter', 'TEST-REVIEW-004']
score = lab.impact_recall(predicted, gold)
assert score['recall_percent'] == 100.0
print('affected controls:', impact.affected_controls)
print(*impact.impacted, sep='\n')
print('impact recall:', score)

## 6 — Separate release reproducibility from current governance
A release snapshot preserves the exact requirement revisions, specialization, exception, evidence IDs, resolver version, and context digest used for a decision. It does not become a new source of policy authority.

In [ ]:
snapshot = lab.release_context(report)
assert snapshot['gate'] == 'ready'
assert snapshot['exceptions'][0]['id'] == 'EXC-014'
assert 'GATEWAY-TELEMETRY-007' in snapshot['evidence']
print(snapshot)

## 7 — Test control effectiveness in an observed window
An implemented review gate is not automatically effective. Compare consequential recommendations with review receipts and authorization evidence, then inject a bypass. The result remains bounded operational evidence, not complete proof.

In [ ]:
import json
runtime_evidence = json.loads((lab.SCENARIO_ROOT / 'reference' / 'runtime-evidence.json').read_text())
baseline_effectiveness = lab.runtime_control_effectiveness(runtime_evidence)
gap_effectiveness = lab.runtime_control_effectiveness({**runtime_evidence, 'legacy_endpoint_bypasses': 37})
assert baseline_effectiveness['status'] == 'effective_in_observed_window'
assert gap_effectiveness['status'] == 'control_gap_detected'
print('baseline:', baseline_effectiveness)
print('failure injection:', gap_effectiveness)

## 8 — Count agent signals without gaming them
Clarification and scope-expansion events can be healthy evidence of bounded autonomy. Keep the counts separate and interpret them with the work context instead of optimizing a single score.

In [ ]:
agent_events = ({'type': 'clarification_request'}, {'type': 'scope_expansion_request'}, {'type': 'scope_expansion_request'}, {'type': 'unauthorized_path_attempt'})
agent_metrics = lab.agent_behavior_metrics(agent_events)
assert agent_metrics['by_type']['scope_expansion_request'] == 2
assert 'score' not in agent_metrics
print(agent_metrics)

## Reflection
Which facts did this local lab truly verify? Which claims still require authenticated publishers, protected repositories, independent approvals, deployed controls, and runtime evidence?